In [1]:
import os

os.chdir("..")

In [16]:
from typing import (
    List,
    Dict,
    Any,
)
import torch
from torchdata.stateful_dataloader import StatefulDataLoader

In [3]:
class StatefulCycleDataLoader(StatefulDataLoader):
    def __call__(self, batch_size: int) -> list[dict[str, Any]]:
        if not hasattr(self, "iterator"):
            self.iterator = iter(self)

        data_list = []
        for _ in range(batch_size):
            try:
                data = next(self.iterator)
            except StopIteration:
                self.iterator = iter(self)
                data = next(self.iterator)
            data_list.append(data)
        return data_list

In [4]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", trust_remote_code=True)

/home/recoverx/astarag/trainer-rl/.venv/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


## SINGLE TURN CHAT MESSAGES

## DATA COLLATOR FOR LANGUAGE MODELING

In [5]:
messages = [
         {"role": "system", "content": "SYSTEM"},
         {"role": "user", "content": "QUESTION"},
         {"role": "assistant", "content": "ANSWER"},
]

print(tokenizer.apply_chat_template(messages,tokenize=False))

<|im_start|>system
SYSTEM<|im_end|>
<|im_start|>user
QUESTION<|im_end|>
<|im_start|>assistant
ANSWER<|im_end|>



In [6]:
tokenizer.apply_chat_template(messages[:-1],tokenize=False, add_generation_prompt=True)

'<|im_start|>system\nSYSTEM<|im_end|>\n<|im_start|>user\nQUESTION<|im_end|>\n<|im_start|>assistant\n'

In [7]:
response_template_ids = tokenizer.encode("\n<|im_start|>assistant\n")
response_template_ids

[198, 151644, 77091, 198]

In [8]:
states = tokenizer.apply_chat_template(messages,tokenize=True)
print(states)

[151644, 8948, 198, 46487, 151645, 198, 151644, 872, 198, 52428, 151645, 198, 151644, 77091, 198, 11692, 39351, 151645, 198]


In [9]:
matches = (
    torch.tensor(states).unfold(0, len(response_template_ids), 1)
    .eq(torch.tensor(response_template_ids))
    .all(dim=1)
)
match_index = torch.nonzero(matches, as_tuple=False).flatten().tolist()[0]
actions =states[match_index+len(response_template_ids):]
action_mask = [0] * (match_index+len(response_template_ids)) + [1] * len(actions)
print(actions)
print(action_mask)

[11692, 39351, 151645, 198]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1]


In [15]:
print(tokenizer.decode([s*a for s,a in zip(states,action_mask) ]))

!!!!!!!!!!!!!!!ANSWER<|im_end|>



## MULTI-TURN TOKENS

In [ ]:
from trainer.datasets import base

SYSTEM_PROMPT = "Let's think step by step"
train_on_what = ["assistant"]

def determine_to_train(role: str) -> bool:
    return role in train_on_what    

messages = [
    base.Message(
            role="system",
            content=SYSTEM_PROMPT,
            train= determine_to_train("system"),
        ),
    base.Message(
        role="user",
        content="How are you?",
        train=determine_to_train("user"),
    ),
    base.Message(
        role="assistant",
        content="ANSWER-1",
        train=determine_to_train("assistant"),
    ),
    base.Message(
        role="user",
        content="Good evening",
        train=determine_to_train("user"),
    ),
    base.Message(
        role="assistant",
        content="ANSWER-2",
        train=determine_to_train("assistant"),
    ),
]

In [56]:
messages_dict = [{"role": m.role, "content": m.content} for m in messages]
messages_dict

[{'role': 'system', 'content': "Let's think step by step"},
 {'role': 'user', 'content': 'How are you?'},
 {'role': 'assistant', 'content': 'ANSWER-1'},
 {'role': 'user', 'content': 'Good evening'},
 {'role': 'assistant', 'content': 'ANSWER-2'}]

In [57]:
print(tokenizer.apply_chat_template(messages_dict,tokenize=False))

<|im_start|>system
Let's think step by step<|im_end|>
<|im_start|>user
How are you?<|im_end|>
<|im_start|>assistant
ANSWER-1<|im_end|>
<|im_start|>user
Good evening<|im_end|>
<|im_start|>assistant
ANSWER-2<|im_end|>



In [68]:
states = tokenizer.apply_chat_template(messages_dict,tokenize=True)

matches = (
    torch.tensor(states).unfold(0, len(response_template_ids), 1)
    .eq(torch.tensor(response_template_ids))
    .all(dim=1)
)
match_index = torch.nonzero(matches, as_tuple=False).flatten().tolist()[-1]
actions = states[match_index+len(response_template_ids):]
action_mask = [0] * (match_index+len(response_template_ids)) + [1] * len(actions)
print(actions)
print(action_mask)

[11692, 39351, 12, 17, 151645, 198]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]


In [69]:
tokenizer.decode(actions)

'ANSWER-2<|im_end|>\n'

In [70]:
messages_dict.append({"role":"user","content":"Good morning"})
messages_dict.append({"role":"assistant","content":"ANSWER-3"})
print(messages_dict)

[{'role': 'system', 'content': "Let's think step by step"}, {'role': 'user', 'content': 'How are you?'}, {'role': 'assistant', 'content': 'ANSWER-1'}, {'role': 'user', 'content': 'Good evening'}, {'role': 'assistant', 'content': 'ANSWER-2'}, {'role': 'user', 'content': 'Good morning'}, {'role': 'assistant', 'content': 'ANSWER-3'}]


In [77]:
states_2 = tokenizer.apply_chat_template(messages_dict,tokenize=False)
print(states_2)

<|im_start|>system
Let's think step by step<|im_end|>
<|im_start|>user
How are you?<|im_end|>
<|im_start|>assistant
ANSWER-1<|im_end|>
<|im_start|>user
Good evening<|im_end|>
<|im_start|>assistant
ANSWER-2<|im_end|>
<|im_start|>user
Good morning<|im_end|>
<|im_start|>assistant
ANSWER-3<|im_end|>



## MULTI-STEP EXAMPLE

In [78]:
multi_turn_messages = [
    base.Message(
            role="system",
            content=SYSTEM_PROMPT,
            train= determine_to_train("system"),
        ),
    base.Message(
        role="user",
        content="How are you?",
        train=determine_to_train("user"),
    ),
    base.Message(
        role="assistant",
        content="ANSWER-1",
        train=determine_to_train("assistant"),
    ),
    base.Message(
        role="user",
        content="Good evening",
        train=determine_to_train("user"),
    ),
    base.Message(
        role="assistant",
        content="ANSWER-2",
        train=determine_to_train("assistant"),
    ),
]

In [79]:
multi_step_messages = [
    base.Message(
            role="system",
            content=SYSTEM_PROMPT,
            train= determine_to_train("system"),
        ),
    base.Message(
        role="user",
        content="How are you?",
        train=determine_to_train("user"),
    ),
    base.Message(
        role="assistant",
        content="ANSWER-1",
        train=determine_to_train("assistant"),
    ),
    base.Message(
        role="assistant",
        content="ANSWER-2",
        train=determine_to_train("assistant"),
    ),
]

In [80]:
def _tokenize_messages(
        messages: List[base.Message], rm: bool = False
    ) -> List[Dict[str, torch.Tensor]]:
        prev_text: str = ""
        states: List[int] = []
        actions: List[int] = []
        action_mask: List[bool] = []
        tensor_dicts: List[Dict[str, torch.Tensor]] = []

        def to_hf_messages(msgs: List[base.Message]) -> List[Dict[str, str]]:
            # HF chat templates expect {"role": ..., "content": ...}
            return [{"role": m.role, "content": m.content} for m in msgs]

        for turn in range(len(messages)):
            is_this_turn_train: bool = messages[turn].train
            is_next_turn_train: bool = (
                turn + 1 < len(messages) and messages[turn + 1].train
            )

            # Skip turns that are neither targets themselves nor part of the prompt
            # for an upcoming target turn. For example, if the current is system prompt
            if not is_this_turn_train and not is_next_turn_train:
                continue

            text: str = tokenizer.apply_chat_template(
                to_hf_messages(messages[: turn + 1]),
                add_generation_prompt=is_next_turn_train,
                tokenize=False,
            )

            if text.startswith(prev_text):
                delta_text_token_ids = tokenizer.encode(
                    text[len(prev_text) :], add_special_tokens=False
                )
                # Tokenize only the delta string to keep token sequence stable.
                delta_text_token_ids_len = len(delta_text_token_ids)
                states.extend(delta_text_token_ids)
                actions.extend(
                    delta_text_token_ids
                    if is_this_turn_train
                    else delta_text_token_ids_len * [0]
                )
                action_mask.extend(delta_text_token_ids_len * [is_this_turn_train])

            else:
                # Prefix broke (template rendering changed). We only allow a reset
                # right before an assistant/train turn (i.e., we are setting up a new prompt).
                assert (
                    is_next_turn_train
                ), "Template prefix broke at an unexpected point (not right before a train turn)."

                tensor_dicts.append(
                    base.get_tensor_dict(
                        states, actions, action_mask, 512, rm
                    )
                )

                states = tokenizer.encode(text, add_special_tokens=False)
                actions = [0] * len(states)
                action_mask = [False] * len(states)

            prev_text = text

        # Finalize last chunk
        tensor_dicts.append(
            base.get_tensor_dict(
                states, actions, action_mask, 512, rm
            )
        )
        return tensor_dicts

multi_turn = _tokenize_messages(multi_turn_messages)

In [ ]:
print(tokenizer.decode(multi_turn[0]['states']))

<|im_start|>system
Let's think step by step<|im_end|>
<|im_start|>user
How are you?<|im_end|>
<|im_start|>assistant
ANSWER-1<|im_end|>
<|im_start|>user
Good evening<|im_end|>
<|im_start|>assistant
ANSWER-2<|im_end|>


In [87]:
print(tokenizer.decode(multi_turn[0]['states'] * multi_turn[0]['action_mask']))

!!!!!!!!!!!!!!!!!!!!!!
ANSWER-1<|im_end|>!!!!!!!!!!
ANSWER-2<|im_end|>


In [89]:
multi_step = _tokenize_messages(multi_step_messages)
print(tokenizer.decode(multi_step[0]['states']))

<|im_start|>system
Let's think step by step<|im_end|>
<|im_start|>user
How are you?<|im_end|>
<|im_start|>assistant
ANSWER-1<|im_end|>
<|im_start|>assistant
ANSWER-2<|im_end|>


In [90]:
print(tokenizer.decode(multi_step[0]['states'] * multi_step[0]['action_mask']))


!!!!!!!!!!!!!!!!!!!!!!
ANSWER-1<|im_end|>
<|im_start|>assistant
ANSWER-2<|im_end|>


In [101]:
multi_step_search_r1_messages = [
    base.Message(
            role="system",
            content=SYSTEM_PROMPT,
            train= determine_to_train("system"),
        ),
    base.Message(
        role="user",
        content="How are you?",
        train=determine_to_train("user"),
    ),
    base.Message(
        role="assistant",
        content="ANSWER-1",
        train=determine_to_train("assistant"),
    ),
    base.Message(
        role="tool",
        content="Retrieved documents",
        train=determine_to_train("tool"),
    ),
    base.Message(
        role="assistant",
        content="ANSWER-2",
        train=determine_to_train("assistant"),
    ),
]
multi_step_search_r1 = _tokenize_messages(multi_step_search_r1_messages)
print(tokenizer.decode(multi_step_search_r1[0]['states']))

<|im_start|>system
Let's think step by step<|im_end|>
<|im_start|>user
How are you?<|im_end|>
<|im_start|>assistant
ANSWER-1<|im_end|>
<|im_start|>user
<tool_response>
Retrieved documents
</tool_response><|im_end|>
<|im_start|>assistant
ANSWER-2<|im_end|>


In [102]:
print(tokenizer.decode(multi_step_search_r1[0]['states'] * multi_step_search_r1[0]['action_mask']))


!!!!!!!!!!!!!!!!!!!!!!
ANSWER-1<|im_end|>!!!!!!!!!!!!!!!!!!!!
ANSWER-2<|im_end|>


## `get_tensor_dict` FUNCTION 

In [121]:
import math
import torch.nn.functional as F
from trainer.utils.seqlen_balance import get_seqlen_balanced_partitions
from omegaconf import OmegaConf
from trainer.datasets import SFTDataset, get_dataloader

def _tensor_dict_to_minibatches(
    tensor_dict: Dict[str, torch.Tensor],
    multiple_of: int,
    max_length_per_dp: int,
) -> List[Dict[str, torch.Tensor]]:
    """
    Pack sequences into minibatches for higher throughput.
    There are two constrains:
      * The number of minibatches must be multiple of `multiple_of`
      * The length of any minibatch cannot exceed `max_length_per_dp`
    To satisfy the second constraint, the number of minibatches must be
    at least `math.ceil(total_length / max_length_per_dp)`.
    Starting from the first multiple of `multiple_of` that is no less
    than the value, we pack sequences into `n_minibatches` minibatches
    and check whether the second constraint is satisfied. If not, we
    increase `n_minibatches` by `multiple_of` (so that the first
    constraint is always satisfied) and repeat the loop.
    """
    # +1 is for the EOS token. EOS mask are zero everywhere except at eos token
    seq_len_list = (tensor_dict["eos_mask"].argmax(-1) + 1).tolist()
    assert max(seq_len_list) <= max_length_per_dp, (
        f"The longest sequence has a length of {max(seq_len_list)},"
        f"which exceeds the maximum length per DP {max_length_per_dp}."
    )
    n_minibatches = math.ceil(sum(seq_len_list) / max_length_per_dp)
    if n_minibatches % multiple_of != 0:
        n_minibatches += multiple_of - n_minibatches % multiple_of

    # Partition sequences into n_minibatches balanced minibatches.
    while True:

        global PAD_SEQUENCES
        if n_minibatches > len(seq_len_list):
            # If we have a small dataset and very large DP size
            # The number of sequences must be no less than `n_minibatches`.
            # If not, we pad the number of sequences to `n_minibatches`.
            # Like for the last minibatch, we might have a lot of padding.
            # how many padding sequences are required?
            PAD_SEQUENCES = n_minibatches - len(seq_len_list)
            for k, v in tensor_dict.items():
                # F.pad(v, (0,0,0,N)) pads rows (batch dimension) by N zeros.
                # So those fake sequences contribute 0 tokens, which won’t violate budgets.
                tensor_dict[k] = F.pad(
                    v, (0, 0, 0, PAD_SEQUENCES), value=0
                )
            # extend the seq_len_list by the number of padding sequences
            # all of them have length 0
            seq_len_list.extend(PAD_SEQUENCES * [0])
        else:
            PAD_SEQUENCES = 0
        # No constraints in the bin packing algorithm
        partitions: List[List[int]] = get_seqlen_balanced_partitions(
            seq_len_list, k_partitions=n_minibatches, equal_size=False
        )
        max_minibatch_length = max(
            [sum([seq_len_list[p] for p in partition]) for partition in partitions]
        )
        # If any minibatch exceeds the budget, they increase n_minibatches (more bins → smaller bins)
        # but in steps of multiple_of so DP divisibility remains true.
        if max_minibatch_length <= max_length_per_dp:
            break
        n_minibatches += multiple_of
        
    global SHUFFLE_INDICES
    SHUFFLE_INDICES = [p for partition in partitions for p in partition]

    return [
        {k: v[partition] for k, v in tensor_dict.items()} for partition in partitions
    ]



config_data = OmegaConf.create({
    "train": {
        "path": "openai/gsm8k",
        "kwargs": {"name": "main", "split": "train"},
        "prompt_key": "question",
        "response_key": "answer",
        "train_on_what": ["assistant"],
        "apply_chat_template": False,
        "system_prompt": "You are a helpful assistant that can answer questions. Let's think step by step.",
        "max_length": 16384,
        "batch_size": 32,
    },
    "test": {
        "path": None,
        "kwargs": {"name": "main", "split": "test"},
        "prompt_key": "question",
        "response_key": "answer",
        "train_on_what": ["assistant"],
        "apply_chat_template": False,
        "system_prompt": "You are a helpful assistant that can answer questions. Let's think step by step.",
        "max_length": 16384,
    },
    "test_ratio": 0.03,
})

train_dataloader, _ = get_dataloader(SFTDataset, config_data, tokenizer)

# Get one batch - this is the tensor_dict passed to sft_step -> _scatter_data -> _tensor_dict_to_minibatches
tensor_dict = next(iter(train_dataloader))

multiple_of = 2  # ddp_size
max_length_per_dp = 1 * 1 * 4096  # cp_size * tp_size * max_length_per_device

minibatches = _tensor_dict_to_minibatches(
    tensor_dict, multiple_of=multiple_of, max_length_per_dp=max_length_per_dp
)
print(f"Batch: {len(tensor_dict['states'])} seqs -> {len(minibatches)} minibatches")

2026-02-14 22:26:48.955 | INFO     | trainer.datasets.base:_load_single:250 - Loading dataset from openai/gsm8k with kwargs {'name': 'main', 'split': 'train'}
2026-02-14 22:26:49.678 | INFO     | trainer.datasets.base:get_dataloader:295 - Loaded 226 train samples and 1 test samples


Batch: 32 seqs -> 2 minibatches


In [122]:
seq_len_list = [254, 184, 240, 312, 169, 209, 138, 139, 178, 166, 238, 188, 182, 184, 264, 262, 191, 265, 305, 158, 223, 185, 151, 318, 305, 176, 141, 223, 211, 190, 222, 116]

shuffle_indices = [1, 4, 6, 7, 8, 10, 11, 14, 15, 16, 18, 20, 21, 22, 23, 28, 0, 2, 3, 5, 9, 12, 13, 17, 19, 24, 25, 26, 27, 29, 30, 31]

first_rank_indices = shuffle_indices[:16]
second_rank_indices = shuffle_indices[16:32]

first_rank_seq_len = [seq_len_list[i] for i in first_rank_indices]
second_rank_seq_len = [seq_len_list[i] for i in second_rank_indices]

print(sum(first_rank_seq_len))
print(sum(second_rank_seq_len))


3344
3343


In [123]:
sum(seq_len_list[:16]), sum(seq_len_list[16:32])

(3307, 3380)

## DPO DATASET

In [ ]:
import math
import torch.nn.functional as F
from trainer.utils.seqlen_balance import get_seqlen_balanced_partitions
from omegaconf import OmegaConf
from trainer.datasets import DPODataset, get_dataloader

def _tensor_dict_to_minibatches(
    tensor_dict: Dict[str, torch.Tensor],
    multiple_of: int,
    max_length_per_dp: int,
    pair: bool,
) -> List[Dict[str, torch.Tensor]]:
    """
    Pack sequences into minibatches for higher throughput.
    There are two constrains:
      * The number of minibatches must be multiple of `multiple_of`
      * The length of any minibatch cannot exceed `max_length_per_dp`
    To satisfy the second constraint, the number of minibatches must be
    at least `math.ceil(total_length / max_length_per_dp)`.
    Starting from the first multiple of `multiple_of` that is no less
    than the value, we pack sequences into `n_minibatches` minibatches
    and check whether the second constraint is satisfied. If not, we
    increase `n_minibatches` by `multiple_of` (so that the first
    constraint is always satisfied) and repeat the loop.
    """
    # +1 is for the EOS token. EOS mask are zero everywhere except at eos token
    seq_len_list = (tensor_dict["eos_mask"].argmax(-1) + 1).tolist()
    if pair:
        # When pair, every two adjacent sequences will be colocated, so
        # their length are summed. This is for DPO
        # [c1,r1,c2,r2] -> [c1+r1,c2+r2]
        seq_len_list = torch.tensor(seq_len_list).view(-1, 2).sum(-1).tolist()
    assert max(seq_len_list) <= max_length_per_dp, (
        f"The longest sequence has a length of {max(seq_len_list)},"
        f"which exceeds the maximum length per DP {max_length_per_dp}."
    )
    n_minibatches = math.ceil(sum(seq_len_list) / max_length_per_dp)
    if n_minibatches % multiple_of != 0:
        n_minibatches += multiple_of - n_minibatches % multiple_of

    # Partition sequences into n_minibatches balanced minibatches.
    while True:

        global PAD_SEQUENCES
        if n_minibatches > len(seq_len_list):
            # If we have a small dataset and very large DP size
            # The number of sequences must be no less than `n_minibatches`.
            # If not, we pad the number of sequences to `n_minibatches`.
            PAD_SEQUENCES = n_minibatches - len(seq_len_list)
            for k, v in tensor_dict.items():
                # F.pad(v, (0,0,0,N)) pads rows (batch dimension) by N zeros.
                # So those fake sequences contribute 0 tokens, which won’t violate budgets.
                tensor_dict[k] = F.pad(
                    v, (0, 0, 0, (2 if pair else 1) * PAD_SEQUENCES), value=0
                )
            seq_len_list.extend(PAD_SEQUENCES * [0])
        else:
            PAD_SEQUENCES = 0

        partitions: List[List[int]] = get_seqlen_balanced_partitions(
            seq_len_list, k_partitions=n_minibatches, equal_size=False
        )
        max_minibatch_length = max(
            [sum([seq_len_list[p] for p in partition]) for partition in partitions]
        )
        # If any minibatch exceeds the budget, they increase n_minibatches (more bins → smaller bins)
        # but in steps of multiple_of so DP divisibility remains true.
        if max_minibatch_length <= max_length_per_dp:
            break
        n_minibatches += multiple_of

    if pair:
        partitions = [
            [_p for p in partition for _p in [2 * p, 2 * p + 1]]
            for partition in partitions
        ]
    global SHUFFLE_INDICES
    SHUFFLE_INDICES = [p for partition in partitions for p in partition]

    return [
        {k: v[partition] for k, v in tensor_dict.items()} for partition in partitions
    ]



config_data = OmegaConf.create({
    "train": {
        "path": "argilla/dpo-mix-7k",
        "kwargs": {"split": "train"},
        "chosen_messages_key": "chosen",
        "rejected_messages_key": "rejected",
        "train_on_what": ["assistant"],
        "apply_chat_template": True,
        "system_prompt": "You are a helpful assistant that can answer questions. Let's think step by step.",
        "max_length": 1024,
        "batch_size": 32,
    },
    "test": {
        "path": "argilla/dpo-mix-7k",
        "kwargs": {"split": "test"},
        "chosen_messages_key": "chosen",
        "rejected_messages_key": "rejected",
        "train_on_what": ["assistant"],
        "apply_chat_template": True,
        "system_prompt": "You are a helpful assistant that can answer questions. Let's think step by step.",
        "max_length": 1024,
    },
})

train_dataloader, _ = get_dataloader(DPODataset, config_data, tokenizer)

# Get one batch - this is the tensor_dict passed to dpo_step -> _scatter_data -> _tensor_dict_to_minibatches
tensor_dict = next(iter(train_dataloader))

multiple_of = 2  # ddp_size
max_length_per_dp = 1 * 1 * 4096  # cp_size * tp_size * max_length_per_device

minibatches = _tensor_dict_to_minibatches(
    tensor_dict, multiple_of=multiple_of, max_length_per_dp=max_length_per_dp, pair=True
)
print(f"Batch: {len(tensor_dict['states'])} seqs -> {len(minibatches)} minibatches")

2026-02-15 15:06:51.119 | INFO     | trainer.datasets.base:_load_single:256 - Loading dataset from argilla/dpo-mix-7k with kwargs {'split': 'train'}


2026-02-15 15:06:51.931 | INFO     | trainer.datasets.base:_load_single:256 - Loading dataset from argilla/dpo-mix-7k with kwargs {'split': 'test'}
2026-02-15 15:06:52.565 | INFO     | trainer.datasets.base:get_dataloader:301 - Loaded 210 train samples and 1 test samples


Batch: 64 seqs -> 10 minibatches


In [ ]:
partitions = [[4, 5, 18, 19, 36, 37, 38, 39, 46, 47], [6, 7, 12, 13, 44, 45], [26, 27, 34, 35, 42, 43], [10, 11, 52, 53, 62, 63], [0, 1, 22, 23, 28, 29], [30, 31, 50, 51, 58, 59], [14, 15, 16, 17, 32, 33], [54, 55, 56, 57], [2, 3, 20, 21, 48, 49, 60, 61], [8, 9, 24, 25, 40, 41]]
